# 06 · Build the agent loop yourself, with step and cost caps
Level L2 · Part 2 · core · ~30 min · builds on 05

## Goal
Write the agent loop yourself: call the model, run the tools it asks for, send the results back, and stop. The loop
stops on an answer or on one of three caps: a **step cap**, a **spend cap** or a **repeated call**. Each stop prints
its reason.

## Why it matters
Lesson 05 made one round trip, and the notebook decided when to stop: after two calls. An agent has no such
notebook. Without a loop control, a model that keeps asking for the same lookup keeps you paying, and the customer
still gets no answer. **Nothing runs in this block.**

```python
while True:
    reply = model(messages)            # every pass is a paid call
    if not reply.tool_requests:
        break                          # the only exit: the model says it is done
    messages += run_tools(reply)       # the model asks for the same lookup forever?
```

One step with 1,000 input and 100 output tokens costs $0.0004; 10,000 stuck steps cost $4.

**What it makes easy:** one place for the limits and a stop reason you can log. **When you don't need it:** a fixed
two-call workflow has nothing to cap.

## The idea
An agent loop is a `while` loop with exits. Each pass, the model either answers (the loop ends) or asks for tools.
Your code checks the caps, runs the tools it asked for, and sends the results back with the model's own turn.

Three stop rules keep it bounded: the **step cap** counts model calls, the **spend cap** adds up the money from each
reply's usage, and **repeated-call detection** stops when the same tool gets the same arguments again. A final answer
is always returned, even if its call went over budget.

Analogy: a cook with a timer, a food budget and a rule against fetching the same ingredient twice.

## Picture
```mermaid
flowchart TD
  Q[Customer question] --> M[Model call: one step]
  M -->|no tool request| A[Stop: answered]
  M -->|tool requests| C{Step cap or spend cap crossed?}
  C -->|yes| S[Stop before any tool runs]
  C -->|no| R{Same call, same arguments, again?}
  R -->|yes| D[Stop: repeated call]
  R -->|no| T[Run the tools, errors become results]
  T --> M
```

## Already know this?
Part 2 (lessons 06–10) turns the loop into an agent you can test and secure. If you can answer all five,
**skip to lesson 10** (the Part's checkpoint):

1. Can you name the three stop rules an agent loop needs, and what each one counts?
2. Can you say why a tool error goes back to the model as readable text, not ending the run?
3. Can you explain a golden set and a task success rate, and score an agent without a model call?
4. Can you say why a ticket or web page is untrusted data, and what least privilege means for a refund tool?
5. Can you list what a shipped Basic agent needs before release: caps, a golden set, safety checks, one run command?

## Step by step
**Step 1 · the tool (no call).** The tool is the one from lesson 05: `lookup_order`, a schema that checks its
argument, and a registry `run_tool` that validates before it runs. The rows are fictional, made for this course.
The loop does not change when you add a tool, because it only looks up the name in the registry.

In [1]:
import json
from collections import Counter
from dataclasses import dataclass, field

from pydantic import BaseModel, ConfigDict, Field

ORDERS = {  # the same fictional rows as lesson 05
    "1093": {"status": "in transit", "days_late": 3},
    "2207": {"status": "delivered", "days_late": 0},
}


def lookup_order(order_id: str) -> dict:
    """Return the status of one order, or an error the model can read."""
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})


print(lookup_order("1093"))

{'status': 'in transit', 'days_late': 3}


The registry maps a name to a function and its schema; `run_tool` validates before it runs.

In [2]:
class OrderQuery(BaseModel):
    model_config = ConfigDict(extra="forbid")
    order_id: str = Field(description="The order number the customer wrote, digits only, such as 1093.")


ORDER_SCHEMA = OrderQuery.model_json_schema()
REGISTRY = {"lookup_order": (lookup_order, OrderQuery)}  # name -> (function, argument schema)


def run_tool(name: str, args: dict) -> dict:
    fn, schema = REGISTRY[name]
    checked = schema.model_validate(args).model_dump()  # refuses bad arguments before the function runs
    return fn(**checked)

**Step 2 · the client (setup).** The key is read from `.env`, never typed here. While a recording replays, a
placeholder is used and no key is sent. `llm_replay` is a course helper, not part of the SDK: it saves every model
call once, so later runs replay the saved answers. The SDK retries busy-server errors (429, 5xx) with backoff, so
the loop does not have to. Each retry is also a request. The model and prices match lesson 02.

In [3]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("06_agent-loop-from-scratch")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"    # the course's default model, the same as lessons 02 to 05
PRICE_IN, PRICE_OUT = 0.25, 1.50   # USD per 1M tokens, as checked for lesson 02 on 2026-10-20
client = genai.Client(  # in your own project keep vertexai=True and pass your own key
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=4, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx
)
TOOL_CONFIG = types.GenerateContentConfig(  # the tool, declared with the schema from step 1
    tools=[types.Tool(function_declarations=[types.FunctionDeclaration(
        name="lookup_order", description="Look up the status of one order by its number.",
        parameters_json_schema=ORDER_SCHEMA)])],
    tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="AUTO")),
    temperature=0,
)

LLM mode: replay (06_agent-loop-from-scratch.yaml)


**Step 3 · the caps and the bookkeeping (no call).** `Caps` holds the three limits. `Run` holds what one task did:
the answer, the stop reason, the steps, the money, and every tool request the model made.

In [4]:
@dataclass
class Caps:
    # max_steps is the step cap: model calls per task. The 6th call can still answer,
    # but its tool requests will not run.
    max_steps: int = 6
    max_usd: float = 0.01      # the spend cap: money per task
    max_same_calls: int = 1    # repeated-call detection: the same call may run this many times


@dataclass
class Run:
    answer: str = ""
    stop: str = ""                                    # why the loop ended
    steps: int = 0                                    # model calls made
    usd: float = 0.0                                  # money spent so far
    asked: list = field(default_factory=list)         # every tool request the model made
    step_usd: list = field(default_factory=list)      # the running cost after each model call


print("default caps:", Caps())

default caps: Caps(max_steps=6, max_usd=0.01, max_same_calls=1)


`cost_of` reads the usage each reply already carries. `over_cap` holds the two number-based stop rules, so the loop
and the examples below use the same check.

In [5]:
def cost_of(resp) -> float:
    u = resp.usage_metadata  # usage the reply already carries: no extra request
    out = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    return (u.prompt_token_count * PRICE_IN + out * PRICE_OUT) / 1_000_000


def over_cap(run: Run, caps: Caps) -> str | None:
    """The two stop rules that use numbers. Repeated calls are checked per request, in the loop."""
    if run.steps >= caps.max_steps:
        return "step cap"
    if run.usd > caps.max_usd:
        return "spend cap"
    return None


print("over_cap after 6 model calls:", over_cap(Run(steps=6), Caps()))

over_cap after 6 model calls: step cap


**Step 4 · the loop.** This is the one loop in the course you write by hand. The SDK gives the calls, the function
parts and the usage. It does not give a spend cap or a repeated-call check (see Under the hood). Each pass is one
model call. If the reply has no tool request, the loop ends with the answer. Otherwise it checks the number caps
before any tool runs, then runs the requests.

In [6]:
def run_requests(requests, counts: Counter, caps: Caps):
    """Run each tool request in order. Returns the results, or None when a call repeats too often."""
    results = []
    for c in requests:
        key = (c.name, json.dumps(dict(c.args), sort_keys=True))
        counts[key] += 1
        if counts[key] > caps.max_same_calls:  # repeated-call detection: stop, do not run it again
            # results of calls that already ran are dropped when we stop
            return None
        try:
            out = run_tool(c.name, dict(c.args))
        except Exception as e:  # a bad call becomes a result the model can read, not a crash
            out = {"error": str(e)}
        results.append(types.Part.from_function_response(name=c.name, response=out))
    return results

def show(run: Run) -> None:
    print(f"stop: {run.stop} · model calls: {run.steps} · spent ${run.usd:.6f}")
    for call in run.asked:
        print("  model asked:", call)
    if run.answer:
        print("answer:", run.answer)

The loop itself: one model call per pass, the same rules for every pass, and `show` prints the stop reason.

In [7]:
def agent_loop(question: str, caps: Caps) -> Run:
    run, counts = Run(), Counter()
    history = [types.Content(role="user", parts=[types.Part(text=question)])]
    while True:
        resp = client.models.generate_content(model=MODEL, contents=history, config=TOOL_CONFIG)  # one step
        run.steps += 1
        run.usd += cost_of(resp)
        run.step_usd.append(round(run.usd, 7))
        turn = resp.candidates[0].content
        requests = [p.function_call for p in turn.parts if p.function_call]
        if not requests:
            run.answer, run.stop = resp.text, "answered"
            return run
        run.asked += [f"{c.name} {json.dumps(dict(c.args), sort_keys=True)}" for c in requests]
        if stop := over_cap(run, caps):  # a cap was crossed: no tool runs after this point
            run.stop = stop
            return run
        results = run_requests(requests, counts, caps)
        if results is None:
            run.stop = "repeated call"
            return run
        history += [turn, types.Content(role="user", parts=results)]  # the model's own turn, then the results

**Step 5 · run 1, the normal case (2 model calls).** The question from lesson 05. The first reply asks for two
lookups, the tools run, and the second reply answers. The stop reason shows `answered`, and each reply's cost is
on the record. The recording counts every request, including the retry. Run 1 = 2 calls + 1 retry after a busy-server
reply (429), so 3 entries; run 2 = 2 calls, so 2 entries; total 5 recorded entries.

In [8]:
QUESTION = "Where are my orders 1093 and 2207? Check both, then answer in two sentences."
run1 = agent_loop(QUESTION, Caps())
show(run1)
print("running cost after each model call:", run1.step_usd)

stop: answered · model calls: 2 · spent $0.000158
  model asked: lookup_order {"order_id": "1093"}
  model asked: lookup_order {"order_id": "2207"}
answer: Order 1093 is currently in transit and is three days late. Order 2207 has been successfully delivered.
running cost after each model call: [8.35e-05, 0.0001582]


**Step 6 · run 2, a model that asks for the same call again (repeated-call detection).** The customer asks the
agent to confirm a status with a second lookup. A model that does what it is told asks for the same call twice.
The loop runs the first request, sees the second one is the same call, and stops without running it. This rule is
strict on purpose: a real system may allow one retry after an error. `llm_replay.stop()` saves the recording here,
after the last model call of this notebook.

In [9]:
PROBE = "Look up order 1093. Then look it up a second time to confirm the status has not changed, and tell me the status."
run2 = agent_loop(PROBE, Caps(max_steps=6))
show(run2)
llm_replay.stop()  # the last model call of this notebook

stop: repeated call · model calls: 2 · spent $0.000113
  model asked: lookup_order {"order_id": "1093"}
  model asked: lookup_order {"order_id": "1093"}


The first lookup ran. The second was blocked before it ran. No third model call was made.

**Step 7 · the step cap and the spend cap, on recorded numbers (no new call).** The two number caps are checks on
totals, so you do not need a new call to see them fire. Apply the same `over_cap` check to run 1's first reply: its
recorded cost, and one model call made. With a step cap of one, the loop would stop after that call, before any
tool runs. With a spend cap of half that cost, it would stop too. With ten times the cost, it would go on.

In [10]:
first = run1.step_usd[0]
after_one_call = Run(steps=1, usd=first)
print(f"cost of model call 1 (recorded in run 1): ${first:.6f}")
print("step cap of 1      ->", over_cap(after_one_call, Caps(max_steps=1)))
print("spend cap, half    ->", over_cap(after_one_call, Caps(max_usd=first / 2)))
print("spend cap, 10 times->", over_cap(after_one_call, Caps(max_usd=first * 10)))

cost of model call 1 (recorded in run 1): $0.000083
step cap of 1      -> step cap
spend cap, half    -> spend cap
spend cap, 10 times-> None


## Under the hood
The SDK can run this loop for you. Pass plain Python functions as tools, and it calls them and sends the results back.
Its only limit is the number of calls, `maximum_remote_calls` (default 10). It has no spend cap and no repeated-call
check, so the loop above is the part you write. Here the setting is built and printed, with no model call.

In [11]:
sdk_loop = types.GenerateContentConfig(
    tools=[lookup_order],  # a plain Python function: the SDK builds the schema from its type hints
    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=6),
)
afc = sdk_loop.automatic_function_calling
print("SDK step cap:", afc.maximum_remote_calls, "· disabled:", bool(afc.disable))
print("fields of the SDK loop config:", list(types.AutomaticFunctionCallingConfig.model_fields))

SDK step cap: 6 · disabled: False
fields of the SDK loop config: ['disable', 'maximum_remote_calls', 'ignore_call_history']


## At work
- Production agents set a step cap, a spend limit per task and a stop reason, and log the reason with the trace. A
  stop that nobody can see looks like a slow answer → lesson 29 (traces) and lesson 33 (cost caps).
- Repeated steps are a named failure: step repetition was 15.7% in the MAST study of multi-agent failures → lesson 25
  (loops, lost context and guards).
- A timeout or a 429 inside the loop needs a retry with backoff and a timeout on each step, not a crash that loses
  the trace → lesson 32 (reliability).

## Common mistakes
1. **A bad argument ends the task.** The model sent a number where the schema wants a string. The schema check
   raises. Without the `try` in `run_requests`, the whole run stops with no answer and no trace.

In [12]:
run_tool("lookup_order", {"order_id": 1093})   # the model sent a number; the schema says string

ValidationError: 1 validation error for OrderQuery
order_id
  Input should be a valid string [type=string_type, input_value=1093, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type

The loop catches the error and sends `{"error": ...}` back, so the model can retry with a string. The error text
names the field, which is what lets the model fix its call. Lesson 07 covers error text in depth.

2. **Checking the spend only once.** Money is spent on each reply, so check after each reply, as `agent_loop` does.
   A check before the first call sees $0 every time and never fires. The check runs after a reply, so it cannot undo
   the call that crossed the line; it stops the tools and the next call.
3. **Counting only tool calls.** A step is a model call. A reply that asks for three tools is still one step.

## Try it
Repeated-call detection uses one rule: the same call with the same arguments may run `max_same_calls` times, and the
next request stops the loop. Write `is_repeat(counts, key, max_same)` that returns `True` when `counts[key]` is more
than `max_same`. Then check it on three cases: a call seen once with `max_same=1`, a call seen twice with
`max_same=1`, and a call seen twice with `max_same=2`. No model call is needed.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
def is_repeat(counts: Counter, key, max_same: int) -> bool:
    return counts[key] > max_same


key = ("lookup_order", '{"order_id": "1093"}')
seen_once = Counter({key: 1})
seen_twice = Counter({key: 2})
print("seen once, max 1:  ", is_repeat(seen_once, key, 1))
print("seen twice, max 1: ", is_repeat(seen_twice, key, 1))
print("seen twice, max 2: ", is_repeat(seen_twice, key, 2))

seen once, max 1:   False
seen twice, max 1:  True
seen twice, max 2:  False


## Recap
- An agent loop stops on an answer, a step cap, a spend cap, or a repeated call. Each stop returns a reason.
- The step cap counts model calls; the spend cap adds up each reply's usage. Both are checked before any tool runs.
- A bad tool call becomes a result the model can read, so one error does not end the task.
- The SDK caps only the number of calls; the spend and repeat rules are yours to write.

**Interview:** "An agent loops forever on one customer. How do you stop it, and what do you log?" Strong answer: "I cap
model calls and spend per task, and stop when the same tool call repeats. Each stop logs its reason, and a tool
error goes back to the model as a result."